# ConsumerVoice: DistilBERT Fine-Tuning for Customer Feedback Sentiment

This notebook fine-tunes `distilbert-base-uncased` on the cleaned, deduplicated, leakage-safe Flipkart customer feedback dataset.

### Target Classification Head:
- **0**: Negative
- **1**: Neutral
- **2**: Positive

### Hardware Recommendation:
- Runtime -> Change runtime type -> **T4 GPU** (or A100/V100 in Google Colab).

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy

import os
import json
import time
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Load Processed Dataset Splits
Upload the 3 files generated by `scripts/prepare_dataset.py`:
- `sentiment_train.csv`
- `sentiment_validation.csv`
- `sentiment_test.csv`

In [ ]:
# If running in Colab, upload the files via file picker or Google Drive:
from google.colab import files
import os

if not os.path.exists('sentiment_train.csv'):
    print("Please upload sentiment_train.csv, sentiment_validation.csv, sentiment_test.csv:")
    uploaded = files.upload()

train_df = pd.read_csv('sentiment_train.csv')
val_df = pd.read_csv('sentiment_validation.csv')
test_df = pd.read_csv('sentiment_test.csv')

print(f"Train samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples: {len(test_df):,}")

## 3. Label Encoding & Class Distributions

In [ ]:
LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
INV_LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}

for df in [train_df, val_df, test_df]:
    df['label'] = df['normalized_sentiment'].map(LABEL_MAP)
    df['text'] = df['cleaned_summary'].fillna('').astype(str)

print("Train class distribution:")
print(train_df['normalized_sentiment'].value_counts())

# Compute class weights to account for dataset imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Computed class weights:", class_weights)

## 4. Tokenization with DistilBERT Tokenizer

In [ ]:
MODEL_NAME = 'distilbert/distilbert-base-uncased'
MAX_LEN = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

## 5. Model Initialization & Weighted Trainer Definition

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=INV_LABEL_MAP,
    label2id=LABEL_MAP
)

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_precision': precision_score(labels, preds, average='macro', zero_division=0),
        'macro_recall': recall_score(labels, preds, average='macro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0)
    }

## 6. Training Configuration & Execution

In [ ]:
training_args = TrainingArguments(
    output_dir='./results_distilbert',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    seed=42,
    fp16=torch.cuda.is_available()
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

start_t = time.time()
trainer.train()
train_time_sec = time.time() - start_t
print(f"Training elapsed time: {train_time_sec:.2f} seconds")

## 7. Independent Untouched Test Split Evaluation

In [ ]:
start_eval = time.time()
test_predictions = trainer.predict(test_ds)
test_latency_ms = ((time.time() - start_eval) / len(test_df)) * 1000.0

y_preds = np.argmax(test_predictions.predictions, axis=-1)
y_trues = test_df['label'].values

acc = accuracy_score(y_trues, y_preds)
macro_p = precision_score(y_trues, y_preds, average='macro', zero_division=0)
macro_r = recall_score(y_trues, y_preds, average='macro', zero_division=0)
macro_f1 = f1_score(y_trues, y_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(y_trues, y_preds, average='weighted', zero_division=0)
cm = confusion_matrix(y_trues, y_preds).tolist()

print("="*60)
print("DISTILBERT TEST SET EVALUATION RESULTS")
print("="*60)
print(f"Accuracy:         {acc:.4f}")
print(f"Macro Precision:  {macro_p:.4f}")
print(f"Macro Recall:     {macro_r:.4f}")
print(f"Macro F1:         {macro_f1:.4f}")
print(f"Weighted F1:      {weighted_f1:.4f}")
print(f"Inference Latency: {test_latency_ms:.2f} ms/sample")
print("\nClassification Report:")
print(classification_report(y_trues, y_preds, target_names=['negative', 'neutral', 'positive']))
print("\nConfusion Matrix:")
print(np.array(cm))

# Structure metrics JSON
transformer_metrics = {
    'accuracy': float(acc),
    'macro_precision': float(macro_p),
    'macro_recall': float(macro_r),
    'macro_f1': float(macro_f1),
    'weighted_f1': float(weighted_f1),
    'training_time_seconds': round(train_time_sec, 2),
    'inference_latency_ms_per_sample': round(test_latency_ms, 2),
    'confusion_matrix': {
        'labels': ['negative', 'neutral', 'positive'],
        'matrix': cm
    }
}
with open('transformer_test_metrics.json', 'w') as f:
    json.dump(transformer_metrics, f, indent=2)

## 8. Export Model Artifacts & Deployment Instructions

In [ ]:
SAVE_DIR = './distilbert_sentiment'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

!zip -r distilbert_sentiment.zip distilbert_sentiment transformer_test_metrics.json
print("\nZip package created: distilbert_sentiment.zip")

# In Colab, trigger automatic download:
files.download('distilbert_sentiment.zip')

print("""
======================================================================
DEPLOYMENT INSTRUCTIONS FOR CONSUMERVOICE:
1. Unzip distilbert_sentiment.zip inside your local ConsumerVoice repo:
   unzip distilbert_sentiment.zip -d backend/saved_models/
2. Move the contents so that distilbert_sentiment is located at:
   backend/saved_models/distilbert_sentiment/
3. Copy transformer_test_metrics.json metrics into backend/saved_models/model_evaluation.json
   under models.transformer_distilbert.
4. ConsumerVoice will automatically detect the checkpoint, perform a health check,
   and allow selecting DistilBERT as the active inference model!
======================================================================
""")